In [72]:
# Loading the necessary imports
import dynamiqs as dq
import jax.numpy as jnp
from pathlib import Path
from scipy.optimize import least_squares
import json
import matplotlib.pyplot as plt


In [73]:
# Building the dictionary of parameters
params = {
    'Delta_a': -14.0,
    'Delta_b': -14.0,
    'Delta_c': 290.0,

    'Xaa': 0.0014,
    'Xbb': 0.0373,
    'Xcc': 164.0,

    'Xab': 0.0072,
    'Xac': 0.671,
    'Xbc': 3.5,

    'kappa_a': 0.0042,
    'kappa_b': 0.984,
    'kappa_c': 0.0207,

    'Ea': 5.6,
    'Eb': 3.0,
    'Ec': 215.0,
}

In [74]:
def exact_iterative_steady_state(params, tol=1e-6, max_iter=150000, mixing=0.05):
    # Unpack parameters
    Ea, Eb, Ec = params['Ea'], params['Eb'], params['Ec']
    Da, Db, Dc = params['Delta_a'], params['Delta_b'], params['Delta_c']
    ka, kb, kc = params['kappa_a']/(2*jnp.pi), params['kappa_b']/(2*jnp.pi), params['kappa_c']/(2*jnp.pi)
    Xcc, Xac, Xbc = params['Xcc'], params['Xac'], params['Xbc']

    # 1. Start with the linear guess (no Kerr shifts)
    na = jnp.abs(Ea / (ka/2 + 1j*Da))**2
    nb = jnp.abs(Eb / (kb/2 + 1j*Db))**2
    nc = jnp.abs(Ec / (kc/2 + 1j*Dc))**2

    # 2. The Self-Consistent Loop
    for i in range(max_iter):
        # Step A: Calculate the new effective detunings using CURRENT populations
        Da_eff = Da - Xac * nc
        Db_eff = Db - Xbc * nc
        Dc_eff = Dc - Xcc * nc - Xac * na - Xbc * nb

        # Step B: Calculate what the populations SHOULD be
        na_calc = jnp.abs(Ea / (ka/2 + 1j*Da_eff))**2
        nb_calc = jnp.abs(Eb / (kb/2 + 1j*Db_eff))**2
        nc_calc = jnp.abs(Ec / (kc/2 + 1j*Dc_eff))**2

        # Step C: Check the largest error/change between current and calculated
        error = jnp.max(jnp.array([
            jnp.abs(na_calc - na),
            jnp.abs(nb_calc - nb),
            jnp.abs(nc_calc - nc)
        ]))

        # If the numbers stopped changing, we found the exact physical root!
        if error < tol:
            print(f"Success! Converged exactly in {i} iterations.")
            break

        # Step D: Apply Damping/Mixing to prevent infinite bouncing
        # We step 5% toward the new value, and keep 95% of the old value
        na = (1 - mixing) * na + mixing * na_calc
        nb = (1 - mixing) * nb + mixing * nb_calc
        nc = (1 - mixing) * nc + mixing * nc_calc
    else:
        print("Warning: Did not perfectly converge. Try lowering the mixing parameter.")

    # 3. Final Complex Amplitude Calculation
    Da_eff = Da - Xac * nc
    Db_eff = Db - Xbc * nc
    Dc_eff = Dc - Xcc * nc - Xac * na - Xbc * nb

    alpha = Ea / (ka/2 + 1j * Da_eff)
    beta  = Eb / (kb/2 + 1j * Db_eff)
    gamma = Ec / (kc/2 + 1j * Dc_eff)

    # Print the perfectly consistent results
    print("=== True Self-Consistent Steady State ===")
    print("Photon Numbers (n):")
    print(f"  Storage (na)  = {na:.7f}")
    print(f"  Readout (nb)  = {nb:.7f}")
    print(f"  Transmon (nc) = {nc:.7f}")
    
    print("\nSquared Amplitudes (|Amplitude|^2) - MUST MATCH ABOVE:")
    print(f"  Storage (|α|^2)  = {jnp.abs(alpha)**2:.7f}")
    print(f"  Readout (|β|^2)  = {jnp.abs(beta)**2:.7f}")
    print(f"  Transmon (|γ|^2) = {jnp.abs(gamma)**2:.7f}")
    
    print("\nComplex Amplitudes (Real + Imag):")
    print(f"  Storage (α)  = {alpha.real:>9.6f} + {alpha.imag:>9.6f}j")
    print(f"  Readout (β)  = {beta.real:>9.6f} + {beta.imag:>9.6f}j")
    print(f"  Transmon (γ) = {gamma.real:>9.6f} + {gamma.imag:>9.6f}j")
    print("=========================================")

    return jnp.array([na, nb, nc]), jnp.array([alpha, beta, gamma])

In [75]:
nums, amps = exact_iterative_steady_state(params)

Success! Converged exactly in 213 iterations.
=== True Self-Consistent Steady State ===
Photon Numbers (n):
  Storage (na)  = 0.1266898
  Readout (nb)  = 0.0169535
  Transmon (nc) = 2.5831003

Squared Amplitudes (|Amplitude|^2) - MUST MATCH ABOVE:
  Storage (|α|^2)  = 0.1266889
  Readout (|β|^2)  = 0.0169528
  Transmon (|γ|^2) = 2.5831003

Complex Amplitudes (Real + Imag):
  Storage (α)  =  0.000008 +  0.355934j
  Readout (β)  =  0.000442 +  0.130202j
  Transmon (γ) =  0.000020 +  1.607203j
